# Synchronise Eye-Tracking & Body-Tracking Data

Loads processed ET/BT CSV pairs for every participant and interpolates BT onto the ET timeline across each participant's full `raw_timestamp` timeline. The output is one merged CSV per participant; `condition_number` and `trial_number` remain labels, but no longer split the synchronization.

## Strategy

| | Eye-tracking | Body-tracking |
|---|---|---|
| File | `001_cleaned_ET.csv` | `001_cleaned_BT.csv` |
| Folder | `data/eye_tracking/processed/` | `data/body_tracking/processed/` |
| Approx. rate | ~200 Hz | ~90 Hz |
| Alignment clock | `raw_timestamp` (ms) | `raw_timestamp` (ms) |

Before synchronisation, cleaned BT tracker duplicates are preferred: e.g. `clean_Waist_pos_y` replaces raw `Waist_pos_y` and is output as `Waist_pos_y`. The same canonical naming is applied to cleaned position and rotation variables.

**Continuous BT columns** (positions, rotations, velocities) → linear interpolation onto ET timestamps across the participant timeline.  
**Discrete BT columns** (`model_name`, `bad_sample_*`, `is_interpolated_*`, etc.) → nearest-neighbour passthrough across the participant timeline.  
BT-derived columns are written with their canonical names, without a `bt_` prefix.

ET rows outside the first/last BT timestamp receive the nearest boundary BT value rather than staying `NaN`, so BT-derived values are populated across the full ET timeline when source values are available.


# 1. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ET_PROCESSED_DIR = Path("S:/projects/legoVR/experiment_05_2026/eye_tracking/processed")
BT_PROCESSED_DIR = Path("S:/projects/legoVR/experiment_05_2026/body_tracking/processed")
OUTPUT_DIR       = Path("S:/projects/legoVR/experiment_05_2026/merged")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GROUP_COLS = ["participant_id", "condition_number", "trial_number"]
SYNC_GROUP_COLS = ["participant_id"]


# 2. Helper Functions

## 2.1 Load

In [2]:
def normalize_pid(participant_id):
    pid = str(participant_id).strip()
    if not pid.isdigit():
        raise ValueError(f"Participant ID must be numeric, got: {participant_id!r}")
    return pid.zfill(3)


def normalize_group_columns(frame, group_cols=GROUP_COLS, frame_name="frame"):
    frame = frame.copy()
    missing = [col for col in group_cols if col not in frame.columns]
    if missing:
        raise KeyError(f"{frame_name} is missing grouping column(s): {missing}")

    frame["participant_id"] = frame["participant_id"].map(normalize_pid)
    for col in [c for c in group_cols if c != "participant_id"]:
        numeric = pd.to_numeric(frame[col], errors="coerce")
        bad = numeric.isna() & frame[col].notna()
        if bad.any():
            examples = frame.loc[bad, col].drop_duplicates().head(5).tolist()
            raise ValueError(f"{frame_name}.{col} contains non-numeric values: {examples}")
        frame[col] = numeric.astype("Int64")

    return frame


def load_cleaned_pair(participant_id,
                      et_dir=ET_PROCESSED_DIR,
                      bt_dir=BT_PROCESSED_DIR):
    pid = normalize_pid(participant_id)
    et_path = Path(et_dir) / f"{pid}_cleaned_ET.csv"
    bt_path = Path(bt_dir) / f"{pid}_cleaned_BT.csv"

    missing = [str(p) for p in (et_path, bt_path) if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing file(s):\n- " + "\n- ".join(missing))

    et_df = normalize_group_columns(pd.read_csv(et_path, low_memory=False), frame_name="ET")
    bt_df = normalize_group_columns(pd.read_csv(bt_path, low_memory=False), frame_name="BT")

    for frame_name, frame in (("ET", et_df), ("BT", bt_df)):
        if "raw_timestamp" not in frame.columns:
            raise KeyError(f"{frame_name} file must contain 'raw_timestamp'.")
        frame["raw_timestamp"] = pd.to_numeric(frame["raw_timestamp"], errors="coerce")

    if "gaze_capture_time" not in et_df.columns:
        raise KeyError("ET file must contain 'gaze_capture_time'.")
    et_df["gaze_capture_time"] = pd.to_numeric(et_df["gaze_capture_time"], errors="coerce")

    print(f"ET  {pid}: {len(et_df):>7,} rows  |  {et_path.name}")
    print(f"BT  {pid}: {len(bt_df):>7,} rows  |  {bt_path.name}")

    return pid, et_path, bt_path, et_df, bt_df


## 2.2 Sampling Rate Summaries

In [3]:
def et_sampling_rate_summary(df):
    t     = pd.to_numeric(df["gaze_capture_time"], errors="coerce").dropna().sort_values()
    dt_ns = t.diff().dropna()
    dt_ns = dt_ns[dt_ns > 0]
    median_dt_ns = dt_ns.median()
    return pd.DataFrame([{
        "stream":                  "ET",
        "time_col":                "gaze_capture_time",
        "n_rows":                  len(t),
        "unique_timestamps":       t.nunique(),
        "median_dt":               median_dt_ns,
        "median_dt_unit":          "ns",
        "median_sampling_rate_hz": 1e9 / median_dt_ns if pd.notna(median_dt_ns) else np.nan,
    }])


def bt_sampling_rate_summary(df):
    t     = pd.to_numeric(df["raw_timestamp"], errors="coerce").dropna().sort_values()
    dt_ms = t.diff().dropna()
    dt_ms = dt_ms[dt_ms > 0]
    median_dt_ms = dt_ms.median()
    return pd.DataFrame([{
        "stream":                  "BT",
        "time_col":                "raw_timestamp",
        "n_rows":                  len(t),
        "unique_timestamps":       t.nunique(),
        "median_dt":               median_dt_ms,
        "median_dt_unit":          "ms",
        "median_sampling_rate_hz": 1000.0 / median_dt_ms if pd.notna(median_dt_ms) else np.nan,
    }])


## 2.3 BT Column Classification

In [4]:
def canonical_bt_column_name(col):
    if not col.startswith("clean_"):
        return col

    candidate = col.removeprefix("clean_")
    if "_pos_" in candidate or "_rot_" in candidate:
        return candidate
    return col


def prepare_bt_columns_for_sync(bt_df):
    """Prefer cleaned tracker columns by renaming them to their raw-style names."""
    bt_clean = bt_df.copy()
    clean_to_canonical = {
        col: canonical_bt_column_name(col)
        for col in bt_clean.columns
        if canonical_bt_column_name(col) != col
    }

    raw_duplicates = [
        canonical
        for canonical in clean_to_canonical.values()
        if canonical in bt_clean.columns
    ]
    if raw_duplicates:
        bt_clean = bt_clean.drop(columns=raw_duplicates)

    bt_clean = bt_clean.rename(columns=clean_to_canonical)

    if clean_to_canonical:
        print(
            f"  Canonicalized cleaned BT cols ({len(clean_to_canonical)}); "
            f"dropped raw duplicate cols ({len(raw_duplicates)})"
        )

    return bt_clean


def get_bt_column_groups(bt_df, et_df=None, group_cols=GROUP_COLS):
    exclude = {"raw_timestamp", *group_cols}

    if et_df is not None:
        duplicate_cols = set(bt_df.columns).intersection(et_df.columns)
        exclude.update(duplicate_cols)

    discrete_passthrough = {
        "model_name", "model_rot_deg",
        "LeftFootArea", "RightFootArea",
        "source_file", "segment_label",
        "sampling_rate_hz", "dt", "dt_ms",
        "participant_id",
    }

    continuous_cols = []
    discrete_cols = []

    for col in bt_df.columns:
        if col in exclude:
            continue

        if col.startswith("bad_sample_") or col.startswith("is_interpolated_"):
            discrete_cols.append(col)
        elif col in discrete_passthrough:
            discrete_cols.append(col)
        elif pd.api.types.is_numeric_dtype(bt_df[col]):
            continuous_cols.append(col)
        else:
            discrete_cols.append(col)

    print(f"  Continuous BT cols ({len(continuous_cols)}): {continuous_cols[:5]}{'...' if len(continuous_cols) > 5 else ''}")
    print(f"  Discrete   BT cols ({len(discrete_cols)}):   {discrete_cols[:5]}{'...' if len(discrete_cols) > 5 else ''}")

    return continuous_cols, discrete_cols

## 2.4 Interpolation

In [5]:
def prepare_bt_segment(bt_segment, continuous_cols, discrete_cols):
    bt_seg = bt_segment.copy()
    bt_seg["raw_timestamp"] = pd.to_numeric(bt_seg["raw_timestamp"], errors="coerce")
    bt_seg = bt_seg.dropna(subset=["raw_timestamp"]).sort_values("raw_timestamp")

    agg_map = {}
    for col in continuous_cols:
        if col in bt_seg.columns:
            agg_map[col] = "mean"
    for col in discrete_cols:
        if col in bt_seg.columns:
            agg_map[col] = "first"

    if not agg_map:
        raise ValueError("No BT columns available for synchronisation.")

    return (
        bt_seg.groupby("raw_timestamp", as_index=False, dropna=False)
        .agg(agg_map)
        .sort_values("raw_timestamp")
        .reset_index(drop=True)
    )


def interpolate_bt_to_et(et_segment, bt_segment, continuous_cols, discrete_cols):
    et_sync = et_segment.copy().sort_values("raw_timestamp").reset_index(drop=True)
    et_t    = pd.to_numeric(et_sync["raw_timestamp"], errors="coerce").to_numpy(dtype=float)

    bt_unique = prepare_bt_segment(bt_segment, continuous_cols, discrete_cols)
    src_t     = bt_unique["raw_timestamp"].to_numpy(dtype=float)

    # nearest-neighbour index for discrete columns
    nearest_idx     = None
    nearest_support = np.zeros(len(et_sync), dtype=bool)

    if len(src_t) >= 1:
        right_idx   = np.searchsorted(src_t, et_t, side="left")
        prev_idx    = np.clip(right_idx - 1, 0, len(src_t) - 1)
        next_idx    = np.clip(right_idx,     0, len(src_t) - 1)
        prev_dist   = np.abs(et_t - src_t[prev_idx])
        next_dist   = np.abs(et_t - src_t[next_idx])
        nearest_idx = np.where(prev_dist <= next_dist, prev_idx, next_idx)
        nearest_support = np.isfinite(et_t)

    # continuous: linear interpolation
    for col in continuous_cols:
        if col not in bt_unique.columns:
            et_sync[col] = np.nan
            continue
        src_v = pd.to_numeric(bt_unique[col], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(src_t) & np.isfinite(src_v)
        if valid.sum() < 2:
            et_sync[col] = np.nan
            continue
        t_v, v_v    = src_t[valid], src_v[valid]
        can_interp  = np.isfinite(et_t)
        interp_vals = np.full(len(et_sync), np.nan, dtype=float)
        interp_vals[can_interp] = np.interp(et_t[can_interp], t_v, v_v)
        et_sync[col] = interp_vals

    # discrete: nearest-neighbour passthrough
    for col in discrete_cols:
        if col not in bt_unique.columns:
            et_sync[col] = np.nan
            continue
        src_series = bt_unique[col]
        if pd.api.types.is_numeric_dtype(src_series):
            carried = np.full(len(et_sync), np.nan, dtype=float)
            if nearest_idx is not None and nearest_support.any():
                src_values = pd.to_numeric(src_series, errors="coerce").to_numpy(dtype=float)
                carried[nearest_support] = src_values[nearest_idx[nearest_support]]
        else:
            carried = np.full(len(et_sync), np.nan, dtype=object)
            if nearest_idx is not None and nearest_support.any():
                src_values = src_series.astype(object).to_numpy()
                carried[nearest_support] = src_values[nearest_idx[nearest_support]]
        et_sync[col] = carried

    return et_sync


def diagnose_et_bt_alignment(et_df, bt_df, group_cols=GROUP_COLS):
    et_norm = normalize_group_columns(et_df, group_cols=group_cols, frame_name="ET")
    bt_norm = normalize_group_columns(bt_df, group_cols=group_cols, frame_name="BT")

    print("\n── Alignment diagnostics ───────────────")
    print("Group key dtypes after normalization:")
    print(pd.DataFrame({
        "ET": et_norm[group_cols].dtypes.astype(str),
        "BT": bt_norm[group_cols].dtypes.astype(str),
    }).to_string())

    et_keys = set(map(tuple, et_norm[group_cols].drop_duplicates().to_numpy()))
    bt_keys = set(map(tuple, bt_norm[group_cols].drop_duplicates().to_numpy()))
    common_keys = et_keys & bt_keys
    print(f"Group keys: ET={len(et_keys)} | BT={len(bt_keys)} | overlap={len(common_keys)}")
    print(f"ET-only keys: {sorted(et_keys - bt_keys)[:10]}")
    print(f"BT-only keys: {sorted(bt_keys - et_keys)[:10]}")

    rows = []
    for key in sorted(common_keys):
        et_mask = np.logical_and.reduce([et_norm[col].eq(value) for col, value in zip(group_cols, key)])
        bt_mask = np.logical_and.reduce([bt_norm[col].eq(value) for col, value in zip(group_cols, key)])
        et_t = pd.to_numeric(et_norm.loc[et_mask, "raw_timestamp"], errors="coerce")
        bt_t = pd.to_numeric(bt_norm.loc[bt_mask, "raw_timestamp"], errors="coerce")
        et_min, et_max = et_t.min(), et_t.max()
        bt_min, bt_max = bt_t.min(), bt_t.max()
        overlap_min = max(et_min, bt_min)
        overlap_max = min(et_max, bt_max)
        has_overlap = pd.notna(overlap_min) and pd.notna(overlap_max) and overlap_min <= overlap_max
        pct_et_inside = ((et_t >= bt_min) & (et_t <= bt_max)).mean() * 100 if len(et_t) else np.nan
        rows.append({
            "participant_id": key[0],
            "condition_number": key[1],
            "trial_number": key[2],
            "et_rows": len(et_t),
            "bt_rows": len(bt_t),
            "has_time_overlap": has_overlap,
            "pct_et_inside_bt_range": round(pct_et_inside, 2) if pd.notna(pct_et_inside) else np.nan,
            "et_start_before_bt_ms": et_min - bt_min,
            "et_end_after_bt_ms": et_max - bt_max,
        })

    report = pd.DataFrame(rows)
    if report.empty:
        print("No overlapping group keys found.")
    else:
        print("Timestamp overlap by group:")
        print(report.to_string(index=False))
        print(f"Mean ET coverage inside BT ranges: {report['pct_et_inside_bt_range'].mean():.2f}%")

    return report


def synchronize_et_bt(et_df, bt_df, group_cols=GROUP_COLS, sync_group_cols=SYNC_GROUP_COLS):
    et_df = normalize_group_columns(et_df, group_cols=group_cols, frame_name="ET")
    bt_df = normalize_group_columns(bt_df, group_cols=group_cols, frame_name="BT")
    bt_df = prepare_bt_columns_for_sync(bt_df)

    continuous_cols, discrete_cols = get_bt_column_groups(bt_df, et_df=et_df, group_cols=sync_group_cols)

    bt_groups = {
        key if isinstance(key, tuple) else (key,): sub.copy()
        for key, sub in bt_df.groupby(sync_group_cols, sort=True, dropna=False)
    }

    empty_bt        = bt_df.iloc[0:0].copy()
    synced_segments = []

    for key, et_segment in et_df.groupby(sync_group_cols, sort=True, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        bt_segment = bt_groups.get(key, empty_bt)
        synced_segments.append(
            interpolate_bt_to_et(et_segment, bt_segment, continuous_cols, discrete_cols)
        )

    return pd.concat(synced_segments, axis=0, ignore_index=True)


## 2.5 Full Build

In [6]:
def discover_participant_ids(et_dir=ET_PROCESSED_DIR, bt_dir=BT_PROCESSED_DIR):
    et_dir = Path(et_dir)
    bt_dir = Path(bt_dir)

    et_ids = {
        path.name.replace("_cleaned_ET.csv", "")
        for path in et_dir.glob("*_cleaned_ET.csv")
    }
    bt_ids = {
        path.name.replace("_cleaned_BT.csv", "")
        for path in bt_dir.glob("*_cleaned_BT.csv")
    }

    participant_ids = sorted(et_ids & bt_ids)
    missing = {
        "missing_et": sorted(bt_ids - et_ids),
        "missing_bt": sorted(et_ids - bt_ids),
    }

    return participant_ids, missing


def get_representative_bt_columns(sync_df, representative_cols=None):
    if representative_cols is not None:
        return [col for col in representative_cols if col in sync_df.columns]

    preferred_tracker_cols = [
        "RightFoot_pos_x", "Waist_pos_x",
        "RightFoot_rot_x", "Waist_rot_y",
    ]
    available = [col for col in preferred_tracker_cols if col in sync_df.columns]
    if available:
        return available

    preferred_metadata_cols = ["model_name", "model_rot_deg"]
    available = [col for col in preferred_metadata_cols if col in sync_df.columns]
    if available:
        return available

    tracker_prefixes = ("RightFoot_", "LeftFoot_", "Waist_", "LeftHand_", "RightHand_")
    metadata_cols = {
        "model_name", "model_rot_deg", "LeftFootArea", "RightFootArea",
        "source_file", "sampling_rate_hz", "dt", "dt_ms",
    }
    return [
        col for col in sync_df.columns
        if col.startswith(tracker_prefixes)
        or col.startswith("bad_sample_")
        or col.startswith("is_interpolated_")
        or col in metadata_cols
    ]


def bt_coverage_by_condition(sync_df, group_col="condition_number", representative_cols=None):
    if group_col not in sync_df.columns:
        raise KeyError(f"Merged dataframe must contain {group_col!r}.")

    representative_cols = get_representative_bt_columns(sync_df, representative_cols=representative_cols)

    if not representative_cols:
        raise ValueError("No BT-derived columns found in merged dataframe.")

    rows = []
    for condition, sub in sync_df.groupby(group_col, sort=True, dropna=False):
        has_bt = sub[representative_cols].notna().any(axis=1)
        rows.append({
            "condition_number": condition,
            "n_rows": int(len(sub)),
            "bt_populated_rows": int(has_bt.sum()),
            "bt_populated_pct": round(float(has_bt.mean() * 100), 2) if len(sub) else np.nan,
            "first_raw_timestamp": sub["raw_timestamp"].min() if "raw_timestamp" in sub.columns else np.nan,
            "last_raw_timestamp": sub["raw_timestamp"].max() if "raw_timestamp" in sub.columns else np.nan,
        })

    return pd.DataFrame(rows)


def build_synchronized_dataframe(participant_id,
                                  et_dir=ET_PROCESSED_DIR,
                                  bt_dir=BT_PROCESSED_DIR,
                                  run_diagnostics=True):
    pid, et_path, bt_path, et_df, bt_df = load_cleaned_pair(
        participant_id, et_dir=et_dir, bt_dir=bt_dir
    )

    et_pre = et_sampling_rate_summary(et_df)
    bt_pre = bt_sampling_rate_summary(bt_df)
    alignment_report = diagnose_et_bt_alignment(et_df, bt_df) if run_diagnostics else pd.DataFrame()

    sync_df  = synchronize_et_bt(et_df, bt_df)
    et_post  = et_sampling_rate_summary(sync_df)
    coverage = bt_coverage_by_condition(sync_df)

    et_pre_hz  = et_pre["median_sampling_rate_hz"].iloc[0]
    et_post_hz = et_post["median_sampling_rate_hz"].iloc[0]

    rate_report = pd.DataFrame([
        {"stage": "pre_sync",  "stream": "ET", "median_sampling_rate_hz": et_pre_hz},
        {"stage": "pre_sync",  "stream": "BT", "median_sampling_rate_hz": bt_pre["median_sampling_rate_hz"].iloc[0]},
        {"stage": "post_sync", "stream": "ET", "median_sampling_rate_hz": et_post_hz},
    ])
    rate_report["rate_diff_from_et_pre_hz"] = (
        rate_report["median_sampling_rate_hz"] - et_pre_hz
    )

    meta = {
        "pid":                   pid,
        "et_path":               et_path,
        "bt_path":               bt_path,
        "et_shape":              et_df.shape,
        "bt_shape":              bt_df.shape,
        "sync_shape":            sync_df.shape,
        "et_pre_hz":             et_pre_hz,
        "et_post_hz":            et_post_hz,
        "alignment_report":      alignment_report,
        "coverage_by_condition": coverage,
    }

    return sync_df, rate_report, meta


def merge_all_participants(participant_ids=None,
                           et_dir=ET_PROCESSED_DIR,
                           bt_dir=BT_PROCESSED_DIR,
                           output_dir=OUTPUT_DIR,
                           run_diagnostics=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    discovered_ids, missing = discover_participant_ids(et_dir=et_dir, bt_dir=bt_dir)
    if participant_ids is None:
        participant_ids = discovered_ids
    else:
        participant_ids = [normalize_pid(pid) for pid in participant_ids]

    summary_rows = []
    coverage_rows = []
    failed = []

    print(f"Found {len(discovered_ids)} participant pair(s): {discovered_ids}")
    if missing["missing_et"]:
        print(f"Missing ET for BT participant(s): {missing['missing_et']}")
    if missing["missing_bt"]:
        print(f"Missing BT for ET participant(s): {missing['missing_bt']}")
    print(f"Merging {len(participant_ids)} participant(s): {participant_ids}\n")

    for pid in participant_ids:
        try:
            sync_df, rate_report, meta = build_synchronized_dataframe(
                pid, et_dir=et_dir, bt_dir=bt_dir, run_diagnostics=run_diagnostics
            )
            out_path = output_dir / f"{meta['pid']}_synced.csv"
            sync_df.to_csv(out_path, index=False)

            coverage = meta["coverage_by_condition"].copy()
            coverage.insert(0, "participant_id", meta["pid"])
            coverage_rows.append(coverage)

            summary_rows.append({
                "participant_id": meta["pid"],
                "et_rows": meta["et_shape"][0],
                "bt_rows": meta["bt_shape"][0],
                "merged_rows": meta["sync_shape"][0],
                "merged_cols": meta["sync_shape"][1],
                "et_pre_hz": meta["et_pre_hz"],
                "et_post_hz": meta["et_post_hz"],
                "output_path": out_path,
            })

            print(f"Saved {meta['pid']} -> {out_path} ({len(sync_df):,} rows, {sync_df.shape[1]} columns)")
            print(coverage.to_string(index=False))
            print()

        except Exception as exc:
            print(f"FAILED {pid}: {exc}")
            failed.append({"participant_id": pid, "error": str(exc)})

    summary_df = pd.DataFrame(summary_rows)
    coverage_df = pd.concat(coverage_rows, ignore_index=True) if coverage_rows else pd.DataFrame()
    failed_df = pd.DataFrame(failed)

    return summary_df, coverage_df, failed_df


# 3. Batch Merge All Participants

Discovers all participants with both processed ET and BT files, merges each pair, and saves one `<participant_id>_synced.csv` file to `OUTPUT_DIR`. The coverage table reports how much of each ET condition has populated BT data after interpolation.

In [ ]:
# Set to a short list like ["001", "002"] for a quick smoke test, or None for all matched participants.
PARTICIPANTS_TO_MERGE = ["004"]

summary_df, coverage_df, failed_df = merge_all_participants(
    participant_ids=PARTICIPANTS_TO_MERGE,
    run_diagnostics=False,
)

print("\nBatch summary:")
print(summary_df.to_string(index=False))

if not failed_df.empty:
    print("\nFailed participants:")
    print(failed_df.to_string(index=False))


Found 1 participant pair(s): ['004']
Merging 1 participant(s): ['004']

ET  004: 667,887 rows  |  004_cleaned_ET.csv
BT  004: 219,476 rows  |  004_cleaned_BT.csv
  Canonicalized cleaned BT cols (15); dropped raw duplicate cols (15)
  Continuous BT cols (53): ['RightFoot_rot_x', 'RightFoot_rot_y', 'RightFoot_rot_z', 'RightFoot_rot_w', 'LeftFoot_rot_x']...
  Discrete   BT cols (16):   ['LeftHand_grabbed_name', 'RightHand_grabbed_name', 'model_rot_deg', 'LeftFootArea', 'RightFootArea']...


In [ ]:
print(summary_df)

In [ ]:
print("\nBT coverage by participant and condition:")
if coverage_df.empty:
    print("No coverage rows available.")
else:
    print(coverage_df.to_string(index=False))


In [ ]:
print(summary_df)

# 4. Optional Single-Participant Debug

Use this only when you want detailed alignment diagnostics for one participant. The batch merge above is the main save workflow.

In [ ]:
DEBUG_PARTICIPANT_ID = '004'  # Example: "001"

if DEBUG_PARTICIPANT_ID is not None:
    debug_sync_df, debug_rate_report, debug_meta = build_synchronized_dataframe(
        DEBUG_PARTICIPANT_ID,
        run_diagnostics=True,
    )

    print("\nShapes")
    print(f"  ET input  : {debug_meta['et_shape']}")
    print(f"  BT input  : {debug_meta['bt_shape']}")
    print(f"  Merged    : {debug_meta['sync_shape']}")
    print("\nSampling rates")
    print(debug_rate_report.to_string(index=False))
    print("\nBT coverage by condition")
    print(debug_meta["coverage_by_condition"].to_string(index=False))

    bt_debug_cols = get_representative_bt_columns(debug_sync_df)
    display(debug_sync_df.loc[debug_sync_df[bt_debug_cols].notna().any(axis=1)].head(3))

    out_path = OUTPUT_DIR / f"{debug_meta['pid']}_synced.csv"
    debug_sync_df.to_csv(out_path, index=False)
    print(f"\nSaved merged participant output: {out_path}")
else:
    print("Set DEBUG_PARTICIPANT_ID to a participant ID to run detailed diagnostics.")


In [ ]:
print(debug_sync_df)

In [ ]:
if DEBUG_PARTICIPANT_ID is not None:
    print("First rows with populated BT data:")
    bt_cols = get_representative_bt_columns(debug_sync_df)
    bt_row_mask = debug_sync_df[bt_cols].notna().any(axis=1)
    display(debug_sync_df.loc[bt_row_mask].head(10))


# 5. Verify Saved Outputs

After running the batch merge, use this optional check to confirm that saved merged CSVs include Condition 0 rows with populated BT columns.

In [ ]:
VERIFY_PARTICIPANTS = ["004"]

verification_rows = []
for pid in VERIFY_PARTICIPANTS:
    path = OUTPUT_DIR / f"{normalize_pid(pid)}_synced.csv"
    if not path.exists():
        verification_rows.append({
            "participant_id": normalize_pid(pid),
            "exists": False,
            "condition0_rows": 0,
            "condition0_bt_populated_rows": 0,
            "condition0_bt_populated_pct": np.nan,
        })
        continue

    df_check = pd.read_csv(path, low_memory=False)
    cond0 = df_check[df_check["condition_number"].eq(0)]
    bt_cols = get_representative_bt_columns(cond0)

    has_bt = cond0[bt_cols].notna().any(axis=1) if bt_cols else pd.Series(False, index=cond0.index)
    verification_rows.append({
        "participant_id": normalize_pid(pid),
        "exists": True,
        "condition0_rows": int(len(cond0)),
        "condition0_bt_populated_rows": int(has_bt.sum()),
        "condition0_bt_populated_pct": round(float(has_bt.mean() * 100), 2) if len(cond0) else np.nan,
    })

verification_df = pd.DataFrame(verification_rows)
print(verification_df.to_string(index=False))
